<a href="https://colab.research.google.com/github/diaoumardia2001-beep/DI-Bootcamp-May/blob/main/Exercises_XP_MCP_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Minimal MCP over STDIO (Student)

Build a tiny MCP server and client that talk over STDIO. This code is supposed to be executed in a local jupyter notebook not Colab's notebook.

## What you'll learn
- How MCP structures hosts/clients/servers and why STDIO is great locally.
- How to register a tool (action) and a resource (read-only context) on a server.
- How to write a client that initializes, lists, and invokes those features.

## Setup
Run the install cell, then restart the runtime if Colab asks. Python 3.10+ required.

In [3]:
!pip install mcp
from mcp.server.fastmcp import FastMCP

# Initialiser un serveur MCP nommé "MonAssistant"
mcp = FastMCP("MonAssistant")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.0 MB/s eta 0:00:00


In [4]:
# Quick verify
!python --version
!mcp --help | head -n 5

Python 3.12.13
                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 MCP development tools                                                          
                                                                                


## A. Server (server.py)
Create a small MCP server named "Demo" with:
- Tool `add(a: int, b: int) -> int` returning the sum.
- Resource template `greeting://{name}` returning "Hello, {name}!".
- Start the STDIO loop in `__main__`.

In [5]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

# Initialisation du serveur FastMCP "Demo"
mcp = FastMCP("Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b

@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    # Démarre la boucle du serveur en utilisant le transport STDIO par défaut
    mcp.run()

Writing server.py


## B. Client (client.py)
Write a client that:
1) Spawns the server via STDIO using the MCP CLI.
2) Initializes a session.
3) Lists resources and tools, printing their names.
4) Reads `greeting://hello` and prints it.
5) Calls tool `add` with a=1, b=7 and prints the result.

In [6]:
%%writefile client.py
import asyncio
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Utilisation de l'exécutable Python actuel pour exécuter le serveur FastMCP via "mcp run"
server_params = StdioServerParameters(
    command="mcp",
    args=["run", "server.py"],
    env=None
)


def extract_content(payload):
    """Best-effort to pull text from MCP responses."""
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        return payload.content
    return str(payload)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Initialisation de la session de communication avec le serveur
            await session.initialize()

            # 1. Lister les ressources disponibles et afficher leurs URIs
            resources_response = await session.list_resources()
            print("--- Available Resources ---")
            for resource in resources_response.resources:
                print(f"URI: {resource.uri} | Name: {resource.name}")
            print()

            # 2. Lister les outils disponibles et afficher leurs noms
            tools_response = await session.list_tools()
            print("--- Available Tools ---")
            for tool in tools_response.tools:
                print(f"Name: {tool.name} | Description: {tool.description}")
            print()

            # 3. Lire la ressource "greeting://Alice" et afficher son contenu
            # Note : "greeting://hello" est également possible, mais le paramètre dynamique attend un nom
            print("--- Reading Resource (greeting://Alice) ---")
            resource_content = await session.read_resource("greeting://Alice")
            print("Content:", extract_content(resource_content))
            print()

            # 4. Appeler l'outil "add" avec les paramètres a=1, b=7 et afficher le résultat
            print("--- Calling Tool (add) ---")
            tool_result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print("Result:", extract_content(tool_result))


if __name__ == "__main__":
    asyncio.run(run())

Writing client.py


## C. Run
One terminal (client spawns server):
```
python client.py
```

Or two terminals:
```
mcp run server.py
python client.py
```

In Colab, run the next cell (client will spawn the server automatically).

In [7]:
# Run the client (spawns the server over STDIO)
!python client.py

[07/16/26 15:23:49] INFO     Processing request of type            ]8;id=623252;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=731390;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListResourcesRequest                               
--- Available Resources ---

                    INFO     Processing request of type            ]8;id=45053;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=915814;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
--- Available Tools ---
Name: add | Description: Return the sum of two integers.

--- Reading Resource (greeting://Alice) ---
                    INFO     Processing request of type            ]8;id=300367;file:///usr/local/lib/python3.12/dist

## Troubleshooting
- `mcp: command not found` ? rerun the install cell or restart runtime.
- Connection closed ? open a second terminal and run `mcp run server.py` to check server errors.
- Type errors ? ensure JSON args are ints for `add`.
Here's a step-by-step breakdown of what happened:

Client and Server Launch: The !python client.py command launched your client script. This script is configured to automatically start the server (server.py) using Standard I/O (STDIO).
Listing Resources: The client requested the server to list available resources. The output --- Available Resources --- indicates that this step was executed. In your server.py, you defined a greeting://{name} resource, but it's not displayed here as an exhaustive list, as 'listing resources' often returns resource 'types' rather than specific instances.
Listing Tools: Next, the client asked the server to list the available tools. The output --- Available Tools --- shows the response:
Name: add | Description: Return the sum of two integers. : This confirms that the add tool you defined in server.py was correctly registered and is available.
Reading greeting://Alice Resource: The client then read the content of the greeting://Alice resource. The server's response was:
Content: Hello, Alice! : This is exactly the expected output from your greet function in server.py, which returns "Hello, {name}!".
Calling add Tool: Finally, the client called the add tool with arguments a=1 and b=7. The server's response was:
Result: [TextContent(type='text', text='8', annotations=None, meta=None)] : The server successfully calculated the sum (1 + 7 = 8) and returned it.
In summary, your client and server communicated successfully: the client was able to discover the server's capabilities (resources and tools), read a specific resource, and call a tool with parameters, and the server responded correctly to all these requests.